# QKD ViT-B/16 Kaggle Run

Run the repository setup cell, then configure the CIFAR-100 dataset. By default, 10% of each CIFAR-100 training class is reserved for dev; the test set is evaluated only once after all tasks finish. The training cell asks `timm` to download the ViT-B/16 pretrained weights; set `QKD_VIT_B16_WEIGHTS` only when you have a local checkpoint file. It forwards the full B0-Inc10 and QGTM configuration to `main.py` and saves per-task checkpoints and development/final-test metrics.

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO_URL = os.environ.get("QKD_REPO_URL", "https://github.com/deety03503/QG_test.git")
REPO_ROOT = Path(os.environ.get("QKD_REPO_ROOT", "/kaggle/working/QG_repro"))

if REPO_ROOT.exists():
    if not (REPO_ROOT / ".git").is_dir():
        raise FileExistsError(f"{REPO_ROOT} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    print(f"Updated repository: {REPO_ROOT}")
else:
    subprocess.run(["git", "clone", "--branch", "Main", GITHUB_REPO_URL, str(REPO_ROOT)], check=True)
    print(f"Cloned repository: {REPO_ROOT}")

os.environ["QKD_REPO_ROOT"] = str(REPO_ROOT)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get('QKD_REPO_ROOT', '/kaggle/working/QG_repro'))
DATA_ROOT = Path(os.environ.get('QKD_CIFAR100_ROOT', '/kaggle/input/cifar_100'))
checkpoint_value = os.environ.get('QKD_VIT_B16_WEIGHTS') or os.environ.get('QKD_VIT_B16_IN21K_WEIGHTS')
CHECKPOINT_PATH = Path(checkpoint_value).expanduser() if checkpoint_value else None
OUTPUT_ROOT = Path(os.environ.get('QKD_OUTPUT_ROOT', '/kaggle/working/qkd_outputs'))

CONFIG = {
    'seed': 1993,
    'batch_size': 32,
    'num_workers': 2,
    'dev_fraction': 0.1,
    'epochs_per_task': 20,
    'learning_rate': 0.05,
    'bottleneck_dim': 64,
    'initial_classes': 0,
    'incremental_classes': 10,
    'svd_dim': 12,
    'num_qubits': 9,
    'circuit_layers': 1,
    'temperature': 1.0,
    'lambda_kd': 1.0,
    'lambda_sparse': 0.05,
    'pretrained_from_timm': True,
}

main_file = REPO_ROOT / 'main.py'
if not main_file.is_file():
    raise FileNotFoundError(f'Cannot find {main_file}; run the clone cell first.')

required_files = [
    REPO_ROOT / 'src' / 'qkd' / 'data' / 'splits.py',
    REPO_ROOT / 'src' / 'qkd' / 'engine' / 'trainer.py',
    REPO_ROOT / 'src' / 'qkd' / 'quantum' / 'circuit.py',
    REPO_ROOT / 'src' / 'qkd' / 'quantum' / 'gating.py',
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_list = ', '.join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f'Incomplete or outdated repository checkout at {REPO_ROOT}. Missing files: '
        f'{missing_list}. Refresh the checkout or point QKD_REPO_ROOT to the latest repository.'
    )
if CHECKPOINT_PATH is not None and not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Configured ViT checkpoint does not exist: {CHECKPOINT_PATH}')
if CHECKPOINT_PATH is None and not CONFIG['pretrained_from_timm']:
    raise ValueError('Provide a checkpoint or enable pretrained_from_timm; random frozen weights are not useful here.')
print(f'Using repository: {REPO_ROOT.resolve()}')
if CHECKPOINT_PATH is not None:
    print(f'Using local checkpoint: {CHECKPOINT_PATH.resolve()}')
else:
    print('Using timm pretrained weights; first run may download weights and requires internet access.')

command = [
    sys.executable, str(main_file),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(CONFIG['seed']),
    '--batch-size', str(CONFIG['batch_size']),
    '--num-workers', str(CONFIG['num_workers']),
    '--dev-fraction', str(CONFIG['dev_fraction']),
    '--epochs-per-task', str(CONFIG['epochs_per_task']),
    '--learning-rate', str(CONFIG['learning_rate']),
    '--bottleneck-dim', str(CONFIG['bottleneck_dim']),
    '--initial-classes', str(CONFIG['initial_classes']),
    '--incremental-classes', str(CONFIG['incremental_classes']),
    '--svd-dim', str(CONFIG['svd_dim']),
    '--num-qubits', str(CONFIG['num_qubits']),
    '--circuit-layers', str(CONFIG['circuit_layers']),
    '--temperature', str(CONFIG['temperature']),
    '--lambda-kd', str(CONFIG['lambda_kd']),
    '--lambda-sparse', str(CONFIG['lambda_sparse']),
]
if CHECKPOINT_PATH is not None:
    command.extend(['--checkpoint-path', str(CHECKPOINT_PATH)])
elif CONFIG['pretrained_from_timm']:
    command.append('--pretrained')
subprocess.run(command, cwd=REPO_ROOT, check=True)